# Greenhouse Slug Discovery

This notebook tests candidate Greenhouse board slugs against:

`https://boards-api.greenhouse.io/v1/boards/{slug}/jobs`

A slug is considered **working** when the API responds `200 OK`.

## Is this possible?

Yes. Greenhouse board URLs are public and can be probed by slug.

- `200` means the board exists and is reachable.
- `404` usually means no board for that slug.

> Note: this cannot guarantee a complete list of *all* companies using Greenhouse.
> It only validates the candidate slugs you provide.

In [37]:
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import Iterable

import requests
import pandas as pd


In [38]:
# --- Configuration ---
BASE_URL = "https://boards-api.greenhouse.io/v1/boards/{slug}/jobs"
MAX_WORKERS = 12
REQUEST_TIMEOUT = 12
SLEEP_JITTER_SEC = (0.0, 0.15)  # polite jitter between calls in each worker

# Seed list from the app's current curated companies.
seed_slugs = [
    "airbnb", "amplitude", "anthropic", "asana", "benchling", "brex", "canva", "cloudflare",
    "coinbase", "datadog", "databricks", "discord", "doordash", "dropbox", "duolingo",
    "figma", "fivetran", "gitlab", "gusto", "hashicorp", "hubspot", "intercom", "lattice",
    "launchdarkly", "linear", "loom", "lyft", "mercury", "miro", "mixpanel", "mongodb",
    "notion", "pagerduty", "palantir", "pinterest", "plaid", "ramp", "reddit", "retool",
    "rippling", "robinhood", "scale", "segment", "snowflake", "stripe", "twilio", "verkada",
    "zendesk", "zoom"
]

# Add your own guesses here (lowercase slug format):
extra_candidates = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/companies.csv")["company"].dropna().tolist()

candidates = sorted(set(seed_slugs + extra_candidates))
print(f"Total candidate slugs: {len(candidates)}")

Total candidate slugs: 1047


In [39]:
def probe_slug(slug: str) -> dict:
    url = BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        # jitter to be polite when running high concurrency
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(r.json().get("jobs", [])) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_slugs(slugs: Iterable[str]) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [40]:
results = scan_slugs(candidates)
results.head(200)

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,2U,200,True,29.0,https://boards-api.greenhouse.io/v1/boards/2U/...,430.3
1,6sense,200,True,29.0,https://boards-api.greenhouse.io/v1/boards/6se...,236.8
2,ASG,200,True,2.0,https://boards-api.greenhouse.io/v1/boards/ASG...,195.5
3,Acquia,200,True,10.0,https://boards-api.greenhouse.io/v1/boards/Acq...,114.3
4,Affirm,200,True,163.0,https://boards-api.greenhouse.io/v1/boards/Aff...,135.8
...,...,...,...,...,...,...
195,Amazon Pay,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ama...,130.1
196,Amazon Web Services,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ama...,157.1
197,Amdocs,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Amd...,121.6
198,Amelia,404,False,NaN,https://boards-api.greenhouse.io/v1/boards/Ame...,139.5


In [41]:
results["url"].tail(200)

847     https://boards-api.greenhouse.io/v1/boards/Spl...
848     https://boards-api.greenhouse.io/v1/boards/Spo...
849     https://boards-api.greenhouse.io/v1/boards/Spr...
850     https://boards-api.greenhouse.io/v1/boards/Spr...
851     https://boards-api.greenhouse.io/v1/boards/Spr...
                              ...                        
1042    https://boards-api.greenhouse.io/v1/boards/sno...
1043    https://boards-api.greenhouse.io/v1/boards/sym...
1044    https://boards-api.greenhouse.io/v1/boards/tes...
1045    https://boards-api.greenhouse.io/v1/boards/zen...
1046    https://boards-api.greenhouse.io/v1/boards/zoo...
Name: url, Length: 200, dtype: object

In [42]:
summary = (
    results.assign(status_group=results["status_code"].fillna(-1))
    .groupby("status_group", dropna=False)
    .size()
    .rename("count")
    .reset_index()
    .sort_values("status_group")
)
summary

,status_group,count
0,200,154
1,404,893


In [43]:
working = results[results["ok"]].copy()
print(f"Working slugs: {len(working)}")
working[["slug", "jobs_count", "status_code", "elapsed_ms"]].head
(50)

Working slugs: 154


50

In [ ]:
# Save only the working slugs — one per line, lowercase, no index, no header.
# Ready to upload with: node scripts/import-slugs.mjs notebooks/greenhouse_slug_scan_results.csv
output_csv = "greenhouse_slug_scan_results.csv"
gh_working = results[results["ok"]].copy()
gh_slugs = (
    gh_working["slug"]
    .str.lower()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
gh_slugs.to_csv(output_csv, index=False, header=False)
print(f"Saved {len(gh_slugs)} slugs to {output_csv}")

---

## Ashby API Check

Testing the same candidate slugs against the Ashby public job board API:

`https://api.ashbyhq.com/posting-api/job-board/{slug}`

- `200` means the company has an Ashby job board at that slug.
- `404` / other means no board found.

> Slugs are case-sensitive in Ashby — this first pass uses lowercase (same as Greenhouse slugs).

In [46]:
ASHBY_BASE_URL = "https://api.ashbyhq.com/posting-api/job-board/{slug}"

def probe_ashby_slug(slug: str) -> dict:
    url = ASHBY_BASE_URL.format(slug=slug)
    t0 = time.perf_counter()
    try:
        r = requests.get(url, timeout=REQUEST_TIMEOUT)
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        time.sleep(random.uniform(*SLEEP_JITTER_SEC))

        jobs = []
        if r.status_code == 200:
            data = r.json()
            jobs = data.get("jobs", [])

        return {
            "slug": slug,
            "status_code": r.status_code,
            "ok": r.status_code == 200,
            "jobs_count": len(jobs) if r.status_code == 200 else None,
            "url": url,
            "elapsed_ms": elapsed_ms,
        }
    except Exception as e:
        elapsed_ms = round((time.perf_counter() - t0) * 1000, 1)
        return {
            "slug": slug,
            "status_code": None,
            "ok": False,
            "jobs_count": None,
            "url": url,
            "elapsed_ms": elapsed_ms,
            "error": str(e),
        }


def scan_ashby_slugs(slugs) -> pd.DataFrame:
    rows = []
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = {pool.submit(probe_ashby_slug, s): s for s in slugs}
        for fut in as_completed(futures):
            rows.append(fut.result())
    df = pd.DataFrame(rows).sort_values(["ok", "status_code", "slug"], ascending=[False, True, True])
    return df.reset_index(drop=True)

In [47]:
ashby_results = scan_ashby_slugs(candidates)

ashby_working = ashby_results[ashby_results["ok"]].copy()
print(f"Working Ashby slugs: {len(ashby_working)} / {len(candidates)}")
ashby_results

Working Ashby slugs: 40 / 1047


,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Applied Intuition,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,948.4
1,Boomi,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,240.5
2,Centerfield,200,True,20.0,https://api.ashbyhq.com/posting-api/job-board/...,426.9
3,Confluent,200,True,58.0,https://api.ashbyhq.com/posting-api/job-board/...,1225.9
4,Crusoe,200,True,306.0,https://api.ashbyhq.com/posting-api/job-board/...,3843.5
...,...,...,...,...,...,...
1042,test IO,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,225.6
1043,twilio,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,215.8
1044,xAI,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/xAI,275.2
1045,zendesk,404,False,NaN,https://api.ashbyhq.com/posting-api/job-board/...,247.9


In [48]:
ashby_working.head()

,slug,status_code,ok,jobs_count,url,elapsed_ms
0,Applied Intuition,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,948.4
1,Boomi,200,True,0.0,https://api.ashbyhq.com/posting-api/job-board/...,240.5
2,Centerfield,200,True,20.0,https://api.ashbyhq.com/posting-api/job-board/...,426.9
3,Confluent,200,True,58.0,https://api.ashbyhq.com/posting-api/job-board/...,1225.9
4,Crusoe,200,True,306.0,https://api.ashbyhq.com/posting-api/job-board/...,3843.5


In [49]:
ashby_clean = ashby_working["slug"]
ashby_clean.to_csv("ashby_slug_scan_results.csv")

In [52]:
csv_greenhouse = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/greenhouse_slug_scan_results.csv")
csv_ashby = pd.read_csv("/Users/giulioelmi/Desktop/job-finder/notebooks/ashby_slug_scan_results.csv")

In [55]:
csv_greenhouse.head()


,slug
0,2U
1,6sense
2,ASG
3,Acquia
4,Affirm


In [54]:
csv_ashby.head()

,Unnamed: 0,slug
0,0,Applied Intuition
1,1,Boomi
2,2,Centerfield
3,3,Confluent
4,4,Crusoe


In [ ]:
# Save only the working Ashby slugs — one per line, lowercase, no index, no header.
# Ready to upload with: node scripts/import-slugs.mjs --table ashby_slugs notebooks/ashby_slug_scan_results.csv
ashby_output_csv = "ashby_slug_scan_results.csv"
ashby_slugs = (
    ashby_working["slug"]
    .str.lower()
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)
ashby_slugs.to_csv(ashby_output_csv, index=False, header=False)
print(f"Saved {len(ashby_slugs)} slugs to {ashby_output_csv}")

---

## Upload Slugs to the Database

Run these commands from the **root of the project** after running the save cells above.

### Greenhouse

```bash
node scripts/import-slugs.mjs notebooks/greenhouse_slug_scan_results.csv
```

### Ashby

```bash
node scripts/import-slugs.mjs --table ashby_slugs notebooks/ashby_slug_scan_results.csv
```

> Re-running either command is safe — existing slugs are skipped (`ON CONFLICT DO NOTHING`).  
> Make sure `.env.local` contains `NEXT_PUBLIC_SUPABASE_URL` and `SUPABASE_SERVICE_ROLE_KEY`.